In [1]:
# Install packages
!pip install pandasgwas pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.8/226.8 kB 13.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pandasgwas: filename=pandasgwas-1.2.2-py3-none-any.whl size=237354 sha256=abb0356b641ccc5e7367e1ad2946200d0ff960638893319aa70ffedea20cdcfc
  Stored in directory: /root/.cache/pip/wheels/e7/14/eb/304c186d1f915eb1bec6a3e62bb4206de300a5db4dde96748f
Successfully built pandasgwas


In [2]:
# Import libraries
import pandas as pd
import os
from pandasgwas import get_variants_by_efo_trait

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Disease queries
disease_queries = {
    "AD":  ["Alzheimer disease"],
    "PD":  ["Parkinson disease"],
    "ALS": ["amyotrophic lateral sclerosis"]
}

print("Disease queries defined:", list(disease_queries.keys()))

Disease queries defined: ['AD', 'PD', 'ALS']


In [8]:
#  Fetch variants for all diseases
print("=== Step 1: GWAS Data Collection using pandasGWAS ===\n")

all_variants = []

for disease, keywords in disease_queries.items():
    print(f"Processing {disease} ...")

    for kw in keywords:
        try:
            result = get_variants_by_efo_trait(efo_trait=kw, interactive=False)

            if result is not None and hasattr(result, 'variants') and not result.variants.empty:
                df = result.variants.copy()
                df['Disease'] = disease

                # Keep important columns
                cols = ['rsId', 'chromosome', 'position', 'functionalClass',
                        'Disease', 'mappedTrait']
                df = df[[c for c in cols if c in df.columns]]

                all_variants.append(df)
                print(f"   Found {len(df)} variants for '{kw}'")
            else:
                print(f"   No variants returned for '{kw}'")

        except Exception as e:
            print(f"   Error with '{kw}': {str(e)[:100]}...")

print("\nVariant collection finished.")

=== Step 1: GWAS Data Collection using pandasGWAS ===

Processing AD ...
   Found 6207 variants for 'Alzheimer disease'
Processing PD ...
   Found 792 variants for 'Parkinson disease'
Processing ALS ...
   Found 166 variants for 'amyotrophic lateral sclerosis'

Variant collection finished.


In [9]:
# Combine all results into one DataFrame
combined = pd.concat(all_variants, ignore_index=True)

print(f"Total variants collected : {len(combined)}")
print(f"Breakdown by disease:")
print(combined['Disease'].value_counts().to_string())

Total variants collected : 7165
Breakdown by disease:
Disease
AD     6207
PD      792
ALS     166


In [10]:
# Remove duplicate rsIDs (per disease)
combined_dedup = combined.drop_duplicates(subset=['rsId', 'Disease'])

print(f"After deduplication      : {len(combined_dedup)} variants")
print(f"Unique rsIDs             : {combined_dedup['rsId'].nunique()}")

After deduplication      : 5860 variants
Unique rsIDs             : 5852


In [11]:
# Preview the data
print(f"Columns: {list(combined_dedup.columns)}")
combined_dedup.head(10)

Columns: ['rsId', 'functionalClass', 'Disease']


,rsId,functionalClass,Disease
0,rs77212406,regulatory_region_variant,AD
1,rs115795926,intergenic_variant,AD
2,rs1761608,non_coding_transcript_exon_variant,AD
3,rs150180355,intergenic_variant,AD
4,rs111609130,intron_variant,AD
5,rs143364530,intron_variant,AD
6,rs139325018,intergenic_variant,AD
7,rs113568679,intron_variant,AD
8,rs140677956,intron_variant,AD
9,rs143982995,intron_variant,AD


In [12]:
# Summary table per disease
summary = (
    combined_dedup
    .groupby('Disease')
    .agg(total_variants=('rsId', 'count'),
         unique_rsIDs   =('rsId', 'nunique'))
    .reset_index()
)

print(" Summary:")
summary

 Summary:


,Disease,total_variants,unique_rsIDs
0,AD,5128,5128
1,ALS,130,130
2,PD,602,602


In [13]:
# Save as CSV
os.makedirs('step1_output', exist_ok=True)

csv_path = 'step1_output/gwas_snps_combined.csv'
combined_dedup.to_csv(csv_path, index=False)

print(f"CSV saved → {csv_path}")

CSV saved → step1_output/gwas_snps_combined.csv


In [14]:
#  Save as Excel (one sheet per disease + summary)
excel_path = 'step1_output/gwas_snps_combined.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    combined_dedup.to_excel(writer, sheet_name='All_SNPs', index=False)
    for disease in ['AD', 'PD', 'ALS']:
        subset = combined_dedup[combined_dedup['Disease'] == disease]
        subset.to_excel(writer, sheet_name=disease, index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f"Excel saved → {excel_path}")

Excel saved → step1_output/gwas_snps_combined.xlsx
